<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/_banners/opim5641_banner.svg" width="100%" alt="OPIM 5641 banner"/>

**Dr. Dave Wanik - Operations and Information Management - University of Connecticut**

---

# Studio 6 · Manage Your Fund (DRAFT)

Your Studio 5 fund has been live on the class leaderboard for a few weeks. Tonight you are its manager, and a manager
answers three questions:

> **1. What happened to my fund, and why?**
> **2. What does the model say today?**
> **3. Do I hold, or do I trade?**

| | Part | What you do | About |
|---|---|---|---|
| 1 | **Load my fund** | paste your record link, Run all | 10 min |
| 2 | **What happened** | the race so far, stock by stock, and what your rules did | 25 min |
| 3 | **Look around** | what the class holds, what the leaders hold, what is in the news | 15 min |
| 4 | **What the model says today** | re-solve Ms. Womack's model with the newest data and see the trades | 20 min |
| 5 | **My decision** | hold, refresh or change your fund, with a reason, and send it to the board | 20 min |

You work on your own fund. Talk with the people around you as much as you like: comparing funds is part of the job.

**This notebook is also your final project.** Tonight is one decision. From now until the market closes on
**Thursday, Dec 3** you can come back, Run all, and change your fund as often as you like. Every change is saved with
its date and your reason. After the Dec 3 close you Run all one last time, and the notebook prints the numbers for
your 2-page report. The report is due **Sunday, Dec 6**, so you have the weekend to write it.

**The race.** The fund worth the most at the Dec 3 close wins the class race, and Dr. Dave writes its manager a
LinkedIn recommendation. Your grade does **not** depend on making money. It depends on how clearly you explain what
happened and what you decided.

## Import modules (download Pyomo, ipopt, Bonmin and yfinance)

Same setup as Studio 5. Run this first, every time. About a minute.

In [ ]:
# install Pyomo, the solvers and yfinance in Colab (idaes-pse brings ipopt and Bonmin)
import sys
import os

if 'google.colab' in sys.modules:
    !pip install -q idaes-pse --pre yfinance
    !idaes get-extensions --to ./bin
    os.environ['PATH'] += ':bin'

%matplotlib inline
from pylab import *
import numpy as np
import pandas as pd
import json
import requests
from pyomo.environ import *
from IPython.display import display
import matplotlib.dates as mdates

import logging
logging.getLogger('pyomo.core').setLevel(logging.ERROR)

# where the setup cell put the solvers
ipopt_path = '/content/bin/ipopt'
bonmin_path = '/content/bin/bonmin'

# Part 1 · Load my fund

When you sent your fund in Studio 5, the receipt printed a line called **your private record**: a long link that ends
in `.json`. It holds your stocks, your rules and your fund's value at every close. Paste it below.

**Lost it?** Ask Dr. Dave: he can look it up.

**No fund on the board yet?** Leave `record_link` blank and pick a Studio 5 list with `my_theme`. The notebook uses an
even split of that list as a stand-in, and in Part 5 you launch your real fund.

In [ ]:
record_link = ''       # paste the "your private record" link from your Studio 5 receipt

# only used when record_link is blank: 'starter' or a theme ('connecticut', 'ai_chips', 'oil_and_gas',
# 'banks_and_payments', 'dividends', 'healthcare', 'food_and_restaurants', 'travel', 'defense_and_aerospace',
# 'utilities_and_power', 'shopping', 'sports_and_entertainment')
my_theme = 'starter'

Run the next cell as it is. It reads your record and the class crowd fund (the average of every student's fund).

In [ ]:
# the Studio 5 starter list and the twelve themes (every company was in the S&P 500 on Dec 31, 2025)
starter = ['AAPL', 'MSFT', 'NVDA', 'GOOGL', 'META', 'JPM', 'GS', 'V', 'JNJ', 'LLY', 'UNH', 'XOM', 'CVX', 'KO', 'WMT',
           'COST', 'AMZN', 'HD', 'CAT', 'LMT']
themes = {
    'connecticut': ['TRV', 'HIG', 'CI', 'SWK', 'RTX', 'OTIS', 'CARR', 'URI', 'SYF', 'HUBB', 'ES', 'IT', 'FDS', 'CHTR', 'XOM', 'PG', 'MSFT', 'UNH'],
    'ai_chips': ['NVDA', 'AMD', 'AVGO', 'MU', 'LRCX', 'AMAT', 'KLAC', 'INTC', 'QCOM', 'TXN', 'ADI', 'ANET', 'MSFT', 'GOOGL', 'META', 'ORCL', 'LLY', 'XOM'],
    'oil_and_gas': ['XOM', 'CVX', 'COP', 'EOG', 'SLB', 'OXY', 'PSX', 'MPC', 'VLO', 'KMI', 'WMB', 'OKE', 'HAL', 'DVN', 'FANG', 'MSFT', 'JNJ'],
    'banks_and_payments': ['JPM', 'BAC', 'WFC', 'C', 'GS', 'MS', 'SCHW', 'V', 'MA', 'AXP', 'PYPL', 'COF', 'BLK', 'USB', 'AAPL', 'UNH', 'CVX'],
    'dividends': ['KO', 'PEP', 'PG', 'JNJ', 'MRK', 'ABBV', 'VZ', 'T', 'DUK', 'SO', 'XOM', 'CVX', 'MMM', 'IBM', 'MO', 'PM', 'CSCO', 'O'],
    'healthcare': ['LLY', 'JNJ', 'UNH', 'ABBV', 'MRK', 'PFE', 'TMO', 'ABT', 'ISRG', 'AMGN', 'GILD', 'CVS', 'MDT', 'BSX', 'SYK', 'AAPL', 'XOM'],
    'food_and_restaurants': ['MCD', 'SBUX', 'CMG', 'YUM', 'DPZ', 'KO', 'PEP', 'MDLZ', 'HSY', 'GIS', 'KHC', 'KDP', 'DRI', 'TSN', 'COST', 'AAPL', 'JNJ', 'XOM'],
    'travel': ['BKNG', 'MAR', 'HLT', 'ABNB', 'EXPE', 'DAL', 'UAL', 'LUV', 'RCL', 'CCL', 'NCLH', 'LVS', 'WYNN', 'MGM', 'UBER', 'AAPL', 'JNJ', 'XOM'],
    'defense_and_aerospace': ['LMT', 'RTX', 'NOC', 'GD', 'BA', 'LHX', 'HII', 'TXT', 'GE', 'HON', 'TDG', 'HWM', 'PLTR', 'LDOS', 'XOM', 'JNJ', 'MSFT'],
    'utilities_and_power': ['NEE', 'DUK', 'SO', 'D', 'AEP', 'EXC', 'SRE', 'XEL', 'PEG', 'ED', 'CEG', 'VST', 'ETR', 'FSLR', 'AES', 'XOM', 'JNJ', 'MSFT'],
    'shopping': ['AMZN', 'WMT', 'COST', 'TGT', 'HD', 'LOW', 'TJX', 'NKE', 'ROST', 'ULTA', 'DG', 'DLTR', 'BBY', 'EBAY', 'AAPL', 'CVS', 'XOM'],
    'sports_and_entertainment': ['DIS', 'NFLX', 'LYV', 'NKE', 'TTWO', 'CMCSA', 'CHTR', 'MAR', 'RCL', 'LVS', 'FOXA', 'HAS', 'GOOGL', 'META', 'AAPL', 'JNJ', 'XOM'],
}

# every S&P 500 company's sector (technology is called 'Technology', as in Studio 5)
members = pd.read_csv('https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/studio6/data/sp500_2025-12-31.csv')
sp500_sectors = {}
for i in range(len(members)):
    sector = members['sector'].iloc[i]
    if sector == 'Information Technology':
        sector = 'Technology'
    sp500_sectors[members['ticker'].iloc[i]] = sector

CROWD_LINK = 'https://storage.googleapis.com/opim5641-leaderboard-f26/crowd.json'
BOARD_URL = 'https://storage.googleapis.com/opim5641-leaderboard-f26/index.html'
STAND_IN_DAY = '2026-11-05'      # a stand-in fund (no record yet) is held from the Studio 5 make-up class

default_rules = {'min_share': 0.02, 'max_share': 0.10, 'must_have_sectors': ['Technology', 'Health Care', 'Energy'],
                 'max_stocks': None}
default_extra = {'sector_cap': None, 'min_stocks': None, 'must_hold': [], 'never_hold': [], 'either_or': [], 'if_then': []}


def read_json(link):
    # a record or the crowd: a web link, or a file in this folder
    if link.strip().startswith('http'):
        return requests.get(link.strip(), timeout=30).json()
    return json.load(open(link.strip()))


# 1. read the record: every version of my fund, oldest first
my_record = None
all_versions = []
if record_link.strip() != '':
    my_record = read_json(record_link)
    for one in my_record['entries']:
        if one.get('fund_kind', 'diversified') != 'concentrated':
            all_versions.append(one)

# 2. my setup (stocks, rules, risk limit) comes from the newest version
has_fund = len(all_versions) > 0
base_stocks = {}
base_rules = dict(default_rules)
base_extra = dict(default_extra)
my_risk_limit = None
my_fund_theme = ''
if has_fund:
    setup = all_versions[-1]
    my_name = my_record.get('nickname', 'my fund')
    my_fund_theme = my_record.get('theme', '')
    my_theme = setup.get('stock_theme') or 'custom'
    if isinstance(setup.get('sectors'), dict):
        for t in setup['sectors']:
            base_stocks[t] = setup['sectors'][t] or sp500_sectors.get(t, 'Other')
    for t in setup['portfolios']['with_rules']:
        if t not in base_stocks:
            base_stocks[t] = sp500_sectors.get(t, 'Other')
    if isinstance(setup.get('rules'), dict):
        for name in setup['rules']:
            base_rules[name] = setup['rules'][name]
    if isinstance(setup.get('extra_rules'), dict):
        for name in setup['extra_rules']:
            base_extra[name] = setup['extra_rules'][name]
    if setup.get('risk_limit') is not None:
        my_risk_limit = float(setup['risk_limit'])
    if base_rules.get('must_have_sectors') is None:
        base_rules['must_have_sectors'] = []
else:
    if my_theme in themes:
        picked = themes[my_theme]
    else:
        my_theme = 'starter'
        picked = starter
    for t in picked:
        base_stocks[t] = sp500_sectors.get(t, 'Other')
    my_name = 'my stand-in fund (%s)' % my_theme

# 3. the versions that have started counting, as (day, mix) pairs for the replay in Part 2
parts = {'with_rules': [], 'no_rules': []}
history = []        # one line per version: the day it counts from, and the reason you gave
for one in all_versions:
    if one.get('counts_from') is None:
        continue
    day = pd.Timestamp(one['counts_from'])
    note = {'day': day, 'reason': one.get('reason', ''), 'notebook': one.get('notebook', '')}
    if len(history) > 0 and history[-1]['day'] == day:
        # two versions before the same close: the later one is the one that counts
        history[-1] = note
        parts['with_rules'][-1] = (day, one['portfolios']['with_rules'])
        parts['no_rules'][-1] = (day, one['portfolios']['no_rules'])
    else:
        history.append(note)
        parts['with_rules'].append((day, one['portfolios']['with_rules']))
        parts['no_rules'].append((day, one['portfolios']['no_rules']))

stand_in = len(history) == 0
if stand_in:
    # no fund has started counting yet: an even split of my list, held from the make-up class, stands in for it
    even_mix = {}
    for t in base_stocks:
        even_mix[t] = 1 / len(base_stocks)
    if has_fund:
        even_mix = all_versions[-1]['portfolios']['with_rules']
    parts['with_rules'].append((pd.Timestamp(STAND_IN_DAY), even_mix))
    parts['no_rules'].append((pd.Timestamp(STAND_IN_DAY), even_mix))
    history.append({'day': pd.Timestamp(STAND_IN_DAY), 'reason': 'stand-in (no fund on the board yet)', 'notebook': ''})

# 4. the class crowd fund
crowd_file = read_json(CROWD_LINK)
crowd_mix = None
class_funds = crowd_file.get('funds', [])
if crowd_file.get('crowd') is not None:
    crowd_mix = crowd_file['crowd']['weights']


def rules_in_words(some_rules, some_extra):
    words = []
    if some_rules.get('max_share') is not None:
        words.append('at most %.0f%% in one stock' % (100 * some_rules['max_share']))
    if some_rules.get('min_share') is not None:
        words.append('at least %.0f%% in any stock held' % (100 * some_rules['min_share']))
    if len(some_rules.get('must_have_sectors') or []) > 0:
        words.append('at least one stock in ' + ', '.join(some_rules['must_have_sectors']))
    if some_rules.get('max_stocks') is not None:
        words.append('no more than %d stocks' % some_rules['max_stocks'])
    for name in some_extra:
        if some_extra[name] not in [None, []]:
            words.append('%s: %s' % (name.replace('_', ' '), some_extra[name]))
    if len(words) == 0:
        return 'no rules'
    return '; '.join(words)


print("Fund:        ", my_name)
print("Stock list:  ", my_theme, "|", len(base_stocks), "stocks to choose from:", ", ".join(sorted(base_stocks)))
print("Rules:       ", rules_in_words(base_rules, base_extra))
if stand_in:
    print("Status:       no fund counting on the board yet, so a stand-in held since Nov 5 is used in Part 2")
else:
    print("Live since:  ", history[0]['day'].strftime('%a %b %d'), "| versions so far:", len(history))
    for k in range(len(history)):
        print("   version %d, from the %s close: %s" % (k + 1, history[k]['day'].strftime('%b %d'), history[k]['reason']))
if crowd_mix is not None:
    print("Class crowd: ", crowd_file['crowd']['members'], "student funds,", len(crowd_mix), "different stocks")

### Daily prices

One download from Yahoo Finance: the daily adjusted closing price of every stock on your list and every stock anyone in
the class holds, plus the S&P 500 (SPY), from September 2021 to the latest close. If Yahoo blocks Colab, the cell uses
the class copy (which starts in October 2024).

In [ ]:
import yfinance as yf

saved_prices = pd.read_csv('https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/studio6/data/daily_prices.csv', index_col=0, parse_dates=True)


def get_prices(tickers):
    # daily closes for these tickers (+ SPY) from Sep 2021; the class copy fills in anything Yahoo will not give us
    wanted = ['SPY']
    for t in tickers:
        if t not in wanted:
            wanted.append(t)
    table = None
    try:
        data = yf.download(wanted, start='2021-09-01', interval='1d', auto_adjust=True, progress=False)
        table = data['Close']
    except Exception as problem:
        table = None
    if table is None or 'SPY' not in table.columns or table['SPY'].notna().sum() < 200 or table.notna().any().mean() < 0.8:
        print("Yahoo did not give us enough stocks, so we are using the class copy instead (it starts in Oct 2024).")
        table = pd.DataFrame(index=saved_prices.index)
    missing = []
    for t in wanted:
        if t not in table.columns or table[t].notna().sum() == 0:
            if t in saved_prices.columns:
                table[t] = saved_prices[t].reindex(table.index)
            else:
                missing.append(t)
    if len(missing) > 0:
        print("No prices for:", missing, "(held as cash)")
    have = []
    for t in wanted:
        if t in table.columns:
            have.append(t)
    table = table[have].sort_index().ffill()
    # today's row is not a close until the market has settled (4:30 PM Eastern), so before then we stop at yesterday
    now_east = pd.Timestamp.now(tz='America/New_York')
    if now_east.hour * 60 + now_east.minute < 16 * 60 + 30:
        table = table[table.index < pd.Timestamp(now_east.date())]
    else:
        table = table[table.index <= pd.Timestamp(now_east.date())]
    return table


everyone = list(base_stocks)
for key in parts:
    for pair in parts[key]:
        everyone = everyone + list(pair[1])
if crowd_mix is not None:
    everyone = everyone + list(crowd_mix)
for fund in class_funds:
    everyone = everyone + list(fund)
prices = get_prices(everyone)
last_close = prices.index[-1]
print(prices.shape[0], "trading days for", prices.shape[1] - 1, "stocks + SPY, through the close of", last_close.date())

# Part 2 · What happened to my fund?

### 2a · The race so far

The leaderboard holds your fund the simple way: \$10,000 at the first close after you sent it, bought in your shares
and then **held**. If you sent a new version later, it started with whatever the fund was worth that day.

The next cell replays that, day by day, for four lines that all start with \$10,000 on your first day:

* **my fund** (with my rules),
* **its no-rules twin** (the same stocks, the same risk limit, no business rules),
* **the class crowd fund** (the average of every student's fund),
* **the S&P 500**.

In [ ]:
def follow(pieces):
    # replay a fund the way the leaderboard does: $10,000 at the first close, held; each later version starts with
    # what the fund was worth at that day's close. Returns the daily line, the dollars each stock added, and what
    # each stock is worth today.
    money = 10000.0
    line = pd.Series(dtype=float)
    added = {}
    holding = {}
    for k in range(len(pieces)):
        first = pieces[k][0]
        mix = pieces[k][1]
        if first > last_close:
            continue
        last = last_close
        if k + 1 < len(pieces) and pieces[k + 1][0] <= last_close:
            last = pieces[k + 1][0]
        window = prices.loc[first:last]
        piece = pd.Series(0.0, index=window.index)
        holding = {}
        for t in mix:
            if t in window.columns and window[t].iloc[0] == window[t].iloc[0]:
                worth = money * mix[t] * window[t] / window[t].iloc[0]
            else:
                worth = pd.Series(money * mix[t], index=window.index)      # no price: held as cash
            piece = piece + worth
            added[t] = added.get(t, 0.0) + float(worth.iloc[-1] - worth.iloc[0])
            holding[t] = float(worth.iloc[-1])
        if len(line) > 0:
            line = pd.concat([line, piece.iloc[1:]])
        else:
            line = piece
        money = float(piece.iloc[-1])
    return line, added, holding


def worst_drop(line):
    # the biggest fall from a high point, in %
    return float(100 * (1 - line / line.cummax()).max())


def in_words(amounts, pattern):
    # a Series as one line of text, for example 'WMT +120, ETR +95'
    pieces = []
    for t in amounts.index:
        pieces.append(pattern % (t, amounts[t]))
    return ", ".join(pieces)


def card(line):
    row = {}
    row['$10,000 became'] = round(float(line.iloc[-1]), 0)
    row['return (%)'] = round(100 * (float(line.iloc[-1]) / float(line.iloc[0]) - 1), 2)
    row['worst drop (%)'] = round(worst_drop(line), 2)
    if len(line) > 1:
        row['worst day (%)'] = round(100 * float(line.pct_change().min()), 2)
        row['best day (%)'] = round(100 * float(line.pct_change().max()), 2)
    return row


my_line, my_added, my_today = follow(parts['with_rules'])
twin_line, twin_added, twin_today = follow(parts['no_rules'])
untouched_line, untouched_added, untouched_today = follow([parts['with_rules'][0]])    # version 1, never changed
first_day = my_line.index[0]
spy_line, spy_added, spy_today = follow([(first_day, {'SPY': 1.0})])
crowd_line = None
if crowd_mix is not None:
    crowd_line, crowd_added, crowd_today = follow([(first_day, crowd_mix)])

my_worth = float(my_line.iloc[-1])
today_share = {}
for t in my_today:
    today_share[t] = my_today[t] / my_worth

lines = {my_name: card(my_line), 'its no-rules twin': card(twin_line), 'S&P 500 (SPY)': card(spy_line)}
if crowd_line is not None:
    lines['class crowd fund'] = card(crowd_line)
scorecard = pd.DataFrame(lines).T

fig, panel = subplots(figsize=(12, 5.5))
panel.plot(spy_line.index, spy_line.values, color='black', linewidth=2, label='S&P 500')
panel.plot(twin_line.index, twin_line.values, color='gray', linewidth=1.5, linestyle='--', label='no-rules twin')
if crowd_line is not None:
    panel.plot(crowd_line.index, crowd_line.values, color='tab:purple', linewidth=1.5, linestyle=':', label='class crowd fund')
panel.plot(my_line.index, my_line.values, color='tab:green', linewidth=3, marker='o', markersize=4, label=my_name)
for k in range(1, len(history)):
    if history[k]['day'] <= last_close:
        panel.axvline(history[k]['day'], color='tab:green', linestyle=':', linewidth=1)
        panel.text(history[k]['day'], panel.get_ylim()[1], ' version %d' % (k + 1), color='tab:green', va='top', fontsize=9)
panel.axhline(10000, color='lightgray', linewidth=1)
panel.set_title('$10,000 since the %s close' % first_day.strftime('%b %d'))
panel.set_ylabel('value ($)')
panel.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
panel.legend(fontsize=9)
show()

print("From the %s close to the %s close (%d trading days):" % (first_day.strftime('%b %d'), last_close.strftime('%b %d'), len(my_line) - 1))
if my_record is not None and not stand_in and len(my_record['values']['with_rules']) > 0:
    print("  check: the leaderboard says $%.0f (through %s), and this replay says $%.0f" % (
        my_record['values']['with_rules'][-1], my_record['values']['dates'][-1], my_worth))
scorecard

### 2b · Stock by stock

A fund's gain is the sum of its stocks' gains. For each stock:

$$\text{dollars added} = \text{dollars you put in} \times \text{how much its price moved}$$

A small share in a stock that moved a lot can matter more than a big share in a stock that did nothing. The table lists
every stock you held, and the bars put them in order. The dollars add up exactly to your fund's gain or loss.

In [ ]:
rows = []
for t in my_added:
    row = {'stock': t, 'sector': base_stocks.get(t) or sp500_sectors.get(t, 'Other')}
    row['share today (%)'] = round(100 * today_share.get(t, 0.0), 1)
    if t in prices.columns and prices.loc[first_day, t] == prices.loc[first_day, t]:
        row['price move since %s (%%)' % first_day.strftime('%b %d')] = round(100 * (prices[t].iloc[-1] / prices.loc[first_day, t] - 1), 1)
    row['dollars added ($)'] = round(my_added[t], 0)
    rows.append(row)
by_stock = pd.DataFrame(rows).set_index('stock').sort_values('dollars added ($)', ascending=False)

by_sector = by_stock.groupby('sector')[['share today (%)', 'dollars added ($)']].sum().sort_values('dollars added ($)', ascending=False)

fig, (left, right) = subplots(1, 2, figsize=(14, max(4.5, 0.32 * len(by_stock))))
ordered = by_stock['dollars added ($)'].sort_values()
colors = []
for t in ordered.index:
    if ordered[t] >= 0:
        colors.append('tab:green')
    else:
        colors.append('black')
left.barh(ordered.index, ordered.values, color=colors)
left.axvline(0, color='gray', linewidth=1)
left.set_title('Dollars each stock added (green) or took away (black)')
left.set_xlabel('dollars')
sector_order = by_sector['dollars added ($)'].sort_values()
colors = []
for name in sector_order.index:
    if sector_order[name] >= 0:
        colors.append('tab:green')
    else:
        colors.append('black')
right.barh(sector_order.index, sector_order.values, color=colors)
right.axvline(0, color='gray', linewidth=1)
right.set_title('The same dollars, by sector')
right.set_xlabel('dollars')
tight_layout()
show()

helpers = by_stock['dollars added ($)'].head(3)
hurters = by_stock['dollars added ($)'].tail(3).sort_values()
print("My fund: $10,000 -> $%.0f, so the stocks added %+.0f dollars in all." % (my_worth, my_worth - 10000))
print("  the three that helped most: " + in_words(helpers, "%s %+.0f"))
print("  the three at the bottom:    " + in_words(hurters, "%s %+.0f"))
total_added = 0
for t in my_added:
    total_added += my_added[t]
print("  check: the stock dollars add up to %+.0f" % total_added)
by_stock

### 2c · What did my rules do?

Your fund and its no-rules twin use the same stocks and the same risk limit. The only difference is your business
rules (the floor, the cap, the sectors). So the gap between the two lines is what the rules earned or cost.

In [ ]:
twin_worth = float(twin_line.iloc[-1])
gap = my_worth - twin_worth
twin_shares = pd.Series(twin_today).sort_values(ascending=False) / twin_worth
print("With my rules: $%.0f. The no-rules twin: $%.0f." % (my_worth, twin_worth))
if gap >= 0:
    print("So far my rules EARNED $%.0f." % gap)
else:
    print("So far my rules COST $%.0f." % (-gap))
print("Worst drop: %.1f%% with my rules, %.1f%% for the twin." % (worst_drop(my_line), worst_drop(twin_line)))
print()
print("My fund holds %d stocks, and the biggest is %.0f%% of it." % (len(my_today), 100 * max(today_share.values())))
print("The twin holds %d: %s" % (len(twin_shares), in_words(100 * twin_shares.head(5), "%s %.0f%%")))
if len(history) > 1 and not stand_in:
    print()
    print("If I had never changed version 1: $%.0f. With my changes: $%.0f. So my changes are worth %+.0f dollars so far." % (
        float(untouched_line.iloc[-1]), my_worth, my_worth - float(untouched_line.iloc[-1])))

### 2d · The days that mattered

A few days usually explain most of a month. The table shows your fund's three best and three worst days, what the
S&P 500 did that day, and which of your stocks moved the most. **Look up one of these dates**: what was in the news?

In [ ]:
if len(my_line) < 4:
    print("Your fund has only been counting for %d trading day(s), so there are not enough days to rank yet." % (len(my_line) - 1))
    big_days = pd.DataFrame()
else:
    my_days = 100 * my_line.pct_change().dropna()
    spy_days = 100 * spy_line.pct_change().dropna()
    priced = []
    for t in my_today:
        if t in prices.columns:
            priced.append(t)
    all_moves = 100 * prices[priced].pct_change(fill_method=None)
    chosen = list(my_days.sort_values(ascending=False).head(3).index) + list(my_days.sort_values().head(3).index)
    rows = []
    for day in chosen:
        mover = all_moves.loc[day].abs().idxmax()
        rows.append({'day': day.strftime('%a %b %d'), 'my fund (%)': round(my_days[day], 2), 'S&P 500 (%)': round(spy_days[day], 2),
                     'my biggest mover': '%s %+.1f%%' % (mover, all_moves.loc[day, mover])})
    big_days = pd.DataFrame(rows).set_index('day')
big_days

### ✍️ Write it down (this is part of your Studio 6 credit)

Double-click this cell and answer in three or four sentences, with numbers from the tables above:

**What happened to my fund since it started, and which two or three stocks explain most of it? Did my rules help?**

*Your answer:*

# Part 3 · Look around

A manager does not decide in a vacuum. Three places to look before you touch your fund.

### 3a · What the class holds

Every student fund, with no names attached. The first table is the stocks the most funds hold. The second is the
class's best and worst movers **that you do not hold**.

In [ ]:
if crowd_mix is None or len(class_funds) == 0:
    print("No student funds on the board yet, so there is nothing to compare with.")
    popular = pd.DataFrame()
    not_mine = pd.DataFrame()
else:
    crowd_day = pd.Timestamp(crowd_file['crowd']['start'])
    move_name = 'move since %s (%%)' % crowd_day.strftime('%b %d')
    held_by = {}
    share_total = {}
    for fund in class_funds:
        for t in fund:
            held_by[t] = held_by.get(t, 0) + 1
            share_total[t] = share_total.get(t, 0.0) + fund[t]
    rows = []
    for t in held_by:
        row = {'stock': t, 'sector': sp500_sectors.get(t, 'Other'), 'funds holding it': held_by[t]}
        row['average share when held (%)'] = round(100 * share_total[t] / held_by[t], 1)
        if t in prices.columns and prices.loc[crowd_day:, t].notna().sum() > 0:
            row[move_name] = round(100 * (prices[t].iloc[-1] / prices.loc[crowd_day:, t].iloc[0] - 1), 1)
        row['in my fund'] = ''
        if t in my_today:
            row['in my fund'] = 'yes'
        rows.append(row)
    class_table = pd.DataFrame(rows).set_index('stock')
    popular = class_table.sort_values(['funds holding it', 'average share when held (%)'], ascending=False).head(12)

    others = class_table[class_table['in my fund'] == ''].dropna(subset=[move_name]).sort_values(move_name, ascending=False)
    not_mine = pd.concat([others.head(5), others.tail(5)])

    same = 0
    for t in today_share:
        same += min(today_share[t], crowd_mix.get(t, 0.0))
    print("%d student funds hold %d different stocks." % (len(class_funds), len(held_by)))
    print("%.0f%% of my fund is the same as the class crowd fund; the other %.0f%% is my own bet." % (100 * same, 100 - 100 * same))
    print()
    print("The stocks the most funds hold:")
    display(popular)
    print("Held by classmates, not by me: their five best and five worst movers")
not_mine

### 3b · What the leaders hold

Open the leaderboard (the next cell prints the link). Click the top two or three funds: each fund's page lists its
stocks and shares. Then ask the person next to you what they hold and why.

* What do the leaders hold that you do not?
* Is one stock or one sector behind most of their lead, or is it spread out?
* Would their fund keep your rules?

In [ ]:
print("The class leaderboard:", BOARD_URL)

### 3c · My stocks this week, and the news

Each stock you hold, over the last week, the last month and 2026 so far, with the biggest movers of the week on top.
**Pick your biggest mover and look it up**: an earnings report, a product, a lawsuit, a rate decision, oil prices.
Then think about the weeks ahead: what is on the calendar between now and Dec 3 that could move your fund?

In [ ]:
rows = []
year_start = prices.loc['2026-01-01':].index[0]
for t in my_today:
    if t not in prices.columns:
        continue
    one = prices[t]
    row = {'stock': t, 'sector': base_stocks.get(t) or sp500_sectors.get(t, 'Other'), 'share today (%)': round(100 * today_share[t], 1)}
    row['last 5 days (%)'] = round(100 * (one.iloc[-1] / one.iloc[-6] - 1), 1)
    row['last month (%)'] = round(100 * (one.iloc[-1] / one.iloc[-22] - 1), 1)
    row['2026 so far (%)'] = round(100 * (one.iloc[-1] / one.loc[year_start] - 1), 1)
    rows.append(row)
this_week = pd.DataFrame(rows).set_index('stock')
this_week = this_week.reindex(this_week['last 5 days (%)'].abs().sort_values(ascending=False).index)
spy_one = prices['SPY']
print("For comparison, the S&P 500: last 5 days %+.1f%%, last month %+.1f%%, 2026 so far %+.1f%%" % (
    100 * (spy_one.iloc[-1] / spy_one.iloc[-6] - 1), 100 * (spy_one.iloc[-1] / spy_one.iloc[-22] - 1),
    100 * (spy_one.iloc[-1] / spy_one.loc[year_start] - 1)))
this_week

### ✍️ Write it down

Double-click and add two notes (one line each is fine):

**One thing I learned from another fund:**

**One thing in the news that matters to my fund:**

# Part 4 · What does the model say today?

In Studio 5 the model read five years of monthly returns, **October 2021 to September 2026**. Since then another
month has closed, and your fund has **drifted**: the stocks that went up are now a bigger share of your money than the
model gave them, and the ones that fell are smaller.

So there are three versions of your fund to compare:

| | What it is |
|---|---|
| **bought as** | the shares the model gave you when you sent the fund |
| **today** | what those shares have drifted to |
| **the model today** | Ms. Womack's model, your rules, your risk limit, solved again with the newest months |

The model is the same one you wrote in Studio 5:

$$
\begin{aligned}
\max \quad & \sum_k \bar r_k \, x_k && \text{(average return)} \\
\text{subject to} \quad & \sum_k x_k = 1 && \text{(invest all of the money)} \\
& \sum_i \sum_j x_i \, \sigma_{ij} \, x_j \le \text{risk limit} && \text{(Ms. Womack's risk)} \\
& \text{floor} \cdot y_k \le x_k \le \text{cap} \cdot y_k, \quad y_k \in \{0, 1\} && \text{(your business rules)}
\end{aligned}
$$

### Step 1 · The newest monthly returns

`fit_years` is how many years of history the model reads. Five is what Studio 5 used. A shorter window trusts recent
months more; that is a choice, and you can try it in Part 5.

In [ ]:
refresh = {
    'fit_years': 5,      # how many years of monthly returns the model reads (Studio 5 used 5)
}

# month-end prices from the daily ones, complete months only (this month is not over yet)
month_end = prices.groupby(prices.index.to_period('M')).last()
month_end = month_end[month_end.index < last_close.to_period('M')]


def monthly_returns(names, years):
    # the last `years` years of monthly returns for these stocks; a stock that is missing months is left out
    have = []
    for t in names:
        if t in month_end.columns:
            have.append(t)
    table = month_end[have].pct_change(fill_method=None).iloc[1:].tail(12 * years)
    complete = []
    for t in have:
        if table[t].isna().sum() == 0:
            complete.append(t)
    return table[complete]


my_stocks = dict(base_stocks)
rules = dict(base_rules)
extra_rules = dict(base_extra)
returns = monthly_returns(list(my_stocks), refresh['fit_years'])
stocks = list(returns.columns)
df_return = returns.mean()
df_cov = returns.cov()

left_out = []
for t in my_stocks:
    if t not in stocks:
        left_out.append(t)
print("The model reads %d months, %s to %s, for %d stocks." % (len(returns), returns.index[0], returns.index[-1], len(stocks)))
if len(left_out) > 0:
    print("Left out (not enough history):", left_out)

### Step 2 · The model (the same `solve_portfolio` as Studio 5)

In [ ]:
def solve_portfolio(risk_limit, use_rules):
    model = ConcreteModel()

    # how much of the money goes in each stock
    model.x = Var(stocks, domain=NonNegativeReals, bounds=(0, 1))
    for s in stocks:
        model.x[s] = 1 / len(stocks)

    # objective: the average return of the portfolio
    obj_expr = 0
    for s in stocks:
        obj_expr += df_return[s] * model.x[s]
    model.portfolio_return = Objective(expr=obj_expr, sense=maximize)

    model.constraints = ConstraintList()

    # invest all of the money
    budget = 0
    for s in stocks:
        budget += model.x[s]
    model.constraints.add(budget == 1)

    # the risk: the weighted covariance matrix, summed
    risk_expr = 0
    for i in stocks:
        for j in stocks:
            risk_expr += model.x[i] * df_cov.loc[i, j] * model.x[j]
    model.constraints.add(risk_expr <= risk_limit)

    if use_rules:
        # the yes/no decision: do we hold stock s?
        model.y = Var(stocks, domain=Binary)
        for s in stocks:
            model.y[s] = 1
        for s in stocks:
            if rules['max_share'] is not None:
                model.constraints.add(model.x[s] <= rules['max_share'] * model.y[s])
            else:
                model.constraints.add(model.x[s] <= model.y[s])
            if rules['min_share'] is not None:
                model.constraints.add(model.x[s] >= rules['min_share'] * model.y[s])
        for sector in rules['must_have_sectors']:
            picks = 0
            count = 0
            for s in stocks:
                if my_stocks.get(s) == sector:
                    picks += model.y[s]
                    count += 1
            if count > 0:    # a sector you have no stocks in is skipped
                model.constraints.add(picks >= 1)
        if rules['max_stocks'] is not None:
            held = 0
            for s in stocks:
                held += model.y[s]
            model.constraints.add(held <= rules['max_stocks'])
        # the extra linking rules
        if extra_rules['sector_cap'] is not None:
            for sector in set(my_stocks.values()):
                in_sector = 0
                count = 0
                for s in stocks:
                    if my_stocks.get(s) == sector:
                        in_sector += model.x[s]
                        count += 1
                if count > 0:
                    model.constraints.add(in_sector <= extra_rules['sector_cap'])
        if extra_rules['min_stocks'] is not None:
            held = 0
            for s in stocks:
                held += model.y[s]
            model.constraints.add(held >= extra_rules['min_stocks'])
        for s in extra_rules['must_hold']:
            if s in stocks:
                model.constraints.add(model.y[s] == 1)
        for s in extra_rules['never_hold']:
            if s in stocks:
                model.constraints.add(model.y[s] == 0)
        for pair in extra_rules['either_or']:
            if pair[0] in stocks and pair[1] in stocks:
                model.constraints.add(model.y[pair[0]] + model.y[pair[1]] <= 1)
        for pair in extra_rules['if_then']:
            if pair[0] in stocks and pair[1] in stocks:
                model.constraints.add(model.y[pair[0]] <= model.y[pair[1]])
        results = SolverFactory('bonmin', executable=bonmin_path).solve(model)
    else:
        results = SolverFactory('ipopt', executable=ipopt_path).solve(model)

    if results.solver.termination_condition != TerminationCondition.optimal:
        return None
    weights = {}
    for s in stocks:
        weights[s] = model.x[s]()
    return {'return': model.portfolio_return(), 'risk': value(risk_expr), 'weights': weights, 'risk_limit': risk_limit}


def even_split_risk():
    # the risk of the same share in every stock: the yardstick from Studio 5, Step 6
    total = 0
    for i in stocks:
        for j in stocks:
            total += df_cov.loc[i, j] / len(stocks) ** 2
    return total


def solve_both(limit):
    # the fund with my rules and its no-rules twin at one risk limit. If nothing fits at that limit (new months can
    # make an old limit too tight), try the even-split risk, then a little more.
    tries = []
    if limit is not None:
        tries.append(limit)
    tries.append(even_split_risk())
    tries.append(1.25 * even_split_risk())
    for one_limit in tries:
        ruled = solve_portfolio(one_limit, use_rules=True)
        if ruled is None:
            continue
        free = solve_portfolio(one_limit, use_rules=False)
        if free is None:
            continue
        return ruled, free, one_limit
    return None, None, None


def shares(result):
    # the stocks the model actually holds, and their shares of the money
    kept = {}
    for s in result['weights']:
        if result['weights'][s] > 0.001:
            kept[s] = round(float(result['weights'][s]), 6)
    return kept


def model_view(mix):
    # what the model thinks of any mix: its average month and its risk, on the months the model just read
    average = 0
    risk = 0
    for i in mix:
        if i in stocks:
            average += mix[i] * df_return[i]
            for j in mix:
                if j in stocks:
                    risk += mix[i] * df_cov.loc[i, j] * mix[j]
    return average, risk


print("solve_portfolio is ready")

### Step 3 · Solve it again, and see the trades

Same stocks, same rules, same risk limit, newest months. The table compares the three versions of your fund, and
**trade (\$)** is what you would buy (+) or sell (−) to move from today to the model's answer.

In [ ]:
target, twin_target, limit_used = solve_both(my_risk_limit)
if target is None:
    raise RuntimeError("The model found no portfolio that keeps these rules. Tell Dr. Dave.")
if my_risk_limit is None:
    print("No risk limit in the record, so the model uses the even-split risk of your list: %.4f" % limit_used)
elif limit_used != my_risk_limit:
    print("Your Studio 5 risk limit (%.4f) is too tight for the newest months, so the model used %.4f." % (my_risk_limit, limit_used))
else:
    print("Risk limit: %.4f, the one you sent in Studio 5." % limit_used)
target_mix = shares(target)
twin_target_mix = shares(twin_target)


def compare(new_mix, new_name):
    # bought as / today / the new mix, side by side, with the trade in dollars
    bought_as = parts['with_rules'][-1][1]
    rows = []
    for t in sorted(set(list(bought_as) + list(today_share) + list(new_mix))):
        row = {'stock': t, 'sector': my_stocks.get(t) or sp500_sectors.get(t, 'Other')}
        row['bought as (%)'] = round(100 * bought_as.get(t, 0.0), 1)
        row['today (%)'] = round(100 * today_share.get(t, 0.0), 1)
        row[new_name + ' (%)'] = round(100 * new_mix.get(t, 0.0), 1)
        row['trade ($)'] = round(my_worth * (new_mix.get(t, 0.0) - today_share.get(t, 0.0)), 0)
        rows.append(row)
    table = pd.DataFrame(rows).set_index('stock').sort_values('trade ($)', ascending=False)
    traded = table['trade ($)'].abs().sum() / 2

    ordered = table.sort_values(new_name + ' (%)')
    fig, panel = subplots(figsize=(11, max(4.5, 0.36 * len(ordered))))
    spots = np.arange(len(ordered))
    panel.barh(spots + 0.2, ordered['today (%)'], height=0.4, color='black', label='today')
    panel.barh(spots - 0.2, ordered[new_name + ' (%)'], height=0.4, color='tab:green', label=new_name)
    panel.set_yticks(spots)
    panel.set_yticklabels(ordered.index)
    panel.set_xlabel('share of my fund (%)')
    panel.set_title('What I hold today (black) vs %s (green)' % new_name)
    panel.legend()
    show()
    return table, traded


trade_table, traded = compare(target_mix, 'the model today')
today_average, today_risk = model_view(today_share)
print("What the model thinks, on the %d months it just read:" % len(returns))
print("  what I hold today:  average month %+.2f%%, risk %.4f (the limit is %.4f)" % (100 * today_average, today_risk, limit_used))
print("  the model today:    average month %+.2f%%, risk %.4f" % (100 * target['return'], target['risk']))
over_cap = []
for t in today_share:
    if rules['max_share'] is not None and today_share[t] > rules['max_share'] + 0.0005:
        over_cap.append('%s %.1f%%' % (t, 100 * today_share[t]))
if len(over_cap) > 0:
    print("  drift: these have grown past my %.0f%% cap: %s" % (100 * rules['max_share'], ", ".join(over_cap)))
print("To move from today to the model's answer I would trade $%.0f, which is %.0f%% of my fund." % (traded, 100 * traded / my_worth))
trade_table

**Read the trades.** A big trade list means one of two things: your fund has drifted a long way, or one new month
changed the model's mind. Ms. Womack's model chases the best past averages, so it can be jumpy. That is worth knowing
before you let it trade for you. A short list means your fund is still close to what the model wants.

# Part 5 · My decision

Three choices. All three are reasonable, and "hold" is a real decision, not a way of skipping the question.

| `choice` | What it does |
|---|---|
| `'hold'` | no trades: the fund stays exactly as it is today |
| `'refresh'` | trade to the model's answer from Part 4 (same stocks, same rules, newest months) |
| `'change'` | change something first (a rule, the stocks on your list, the history the model reads, the risk limit), then solve again |

Then write **your reason** in your own words: one or two sentences. Use what you found tonight: a number from Part 2,
something another fund did, something in the news, the size of the trade list.

`my_change` is only read when your choice is `'change'`. Anything you leave as `None` or `[]` stays as it is now.

In [ ]:
decision = {
    'choice': 'hold',      # 'hold', 'refresh' or 'change'
    'reason': '',          # one or two sentences, in your own words (the board keeps the first 180 characters)
}

my_change = {
    # my rules
    'max_share': None,           # e.g. 0.15: at most 15% in one stock
    'min_share': None,           # e.g. 0.03: at least 3% in any stock I hold
    'max_stocks': None,          # e.g. 8: hold no more than 8 stocks
    'must_have_sectors': None,   # e.g. ['Technology', 'Energy'], or [] for no sector rule
    'never_hold': [],            # e.g. ['TSLA']: the model may not buy these
    'must_hold': [],             # e.g. ['WMT']: the model must hold these
    # my stock list
    'add_stocks': [],            # e.g. ['NVDA', 'COST']: add these tickers to my list
    'drop_stocks': [],           # e.g. ['PNC']: take these off my list
    # what the model reads, and how much risk it may take
    'fit_years': None,           # e.g. 3: read the last 3 years instead of 5
    'risk_times': None,          # e.g. 0.8 for 20% less risk than my limit, 1.2 for 20% more
}

Run the next cell. It works out the fund your decision leads to and shows it next to what you hold today. Change
your choice and run both cells again as often as you like: nothing is sent until the last cell.

In [ ]:
new_mix = None
new_twin = None
new_limit = limit_used

if decision['choice'] == 'hold':
    # no trades: tomorrow's fund is today's fund
    new_mix = {}
    for t in today_share:
        new_mix[t] = round(today_share[t], 6)
    twin_worth_now = float(twin_line.iloc[-1])
    new_twin = {}
    for t in twin_today:
        new_twin[t] = round(twin_today[t] / twin_worth_now, 6)
    if stand_in:
        print("You have no fund on the board yet, so there is nothing to hold. Choose 'refresh' or 'change' to launch one.")
        new_mix = None

elif decision['choice'] == 'refresh':
    # the model's answer from Part 4
    new_mix = target_mix
    new_twin = twin_target_mix

elif decision['choice'] == 'change':
    # start again from the fund as it was loaded, so running this cell twice does not change things twice
    my_stocks = dict(base_stocks)
    rules = dict(base_rules)
    extra_rules = dict(base_extra)

    # 1. my stock list
    for t in my_change['drop_stocks']:
        if t.strip().upper() in my_stocks:
            del my_stocks[t.strip().upper()]
    to_add = []
    for t in my_change['add_stocks']:
        t = t.strip().upper()
        if t not in my_stocks:
            my_stocks[t] = sp500_sectors.get(t, 'Other')
        if t not in prices.columns:
            to_add.append(t)
    if len(to_add) > 0:
        more = get_prices(to_add)
        for t in to_add:
            if t in more.columns:
                prices[t] = more[t].reindex(prices.index).ffill()
        month_end = prices.groupby(prices.index.to_period('M')).last()
        month_end = month_end[month_end.index < last_close.to_period('M')]

    # 2. my rules
    for name in ['max_share', 'min_share', 'max_stocks', 'must_have_sectors']:
        if my_change[name] is not None:
            rules[name] = my_change[name]
    if len(my_change['never_hold']) > 0:
        extra_rules['never_hold'] = list(my_change['never_hold'])
    if len(my_change['must_hold']) > 0:
        extra_rules['must_hold'] = list(my_change['must_hold'])

    # 3. what the model reads, and how much risk it may take
    years = refresh['fit_years']
    if my_change['fit_years'] is not None:
        years = my_change['fit_years']
    returns = monthly_returns(list(my_stocks), years)
    stocks = list(returns.columns)
    df_return = returns.mean()
    df_cov = returns.cov()
    ask_limit = my_risk_limit
    if my_change['risk_times'] is not None and my_risk_limit is not None:
        ask_limit = my_risk_limit * my_change['risk_times']
    elif my_change['risk_times'] is not None:
        ask_limit = even_split_risk() * my_change['risk_times']

    # 4. solve
    print("The model reads %d months (%s to %s) for %d stocks." % (len(returns), returns.index[0], returns.index[-1], len(stocks)))
    print("My rules now:", rules_in_words(rules, extra_rules))
    changed, changed_twin, new_limit = solve_both(ask_limit)
    if changed is None:
        print("No portfolio keeps all of these rules. Loosen a rule, add stocks, or raise risk_times, then run this cell again.")
    else:
        if ask_limit is not None and new_limit != ask_limit:
            print("The risk limit you asked for (%.4f) is too tight for these rules, so the model used %.4f." % (ask_limit, new_limit))
        new_mix = shares(changed)
        new_twin = shares(changed_twin)

else:
    print("decision['choice'] should be 'hold', 'refresh' or 'change'.")

if new_mix is not None:
    decision_table, decision_traded = compare(new_mix, 'after my decision')
    new_average, new_risk = model_view(new_mix)
    print("My decision: %s. I trade $%.0f (%.0f%% of my fund) and hold %d stocks." % (
        decision['choice'], decision_traded, 100 * decision_traded / my_worth, len(new_mix)))
    print("On the months the model read: average month %+.2f%%, risk %.4f." % (100 * new_average, new_risk))
    display(decision_table)

### Send it to the leaderboard

Fill in your NetID and the class code (Dr. Dave writes it on the board), then run the next two cells. Your decision is
saved with tonight's date and your reason, and it counts from the next market close. Your fund keeps its value: a new
version starts with whatever the fund is worth at that close.

**Sending "hold" is worth doing.** It puts tonight's decision and your reason in your fund's history, which is what
your final report is about.

In [ ]:
netid = ""            # your UConn NetID, for example abc12345
class_code = ""       # Dr. Dave writes it on the board
nickname = ""         # leave blank to keep your fund's name (fill it in if this is your first fund)

In [ ]:
SUBMIT_URL = 'https://us-central1-opim5641-portfolio-f26.cloudfunctions.net/submit-portfolio'

send_name = nickname.strip()
if send_name == '' and has_fund:
    send_name = my_name
send_theme = my_fund_theme
if send_theme == '':
    send_theme = my_theme.replace('_', ' ')

missing = []
if netid.strip() == '':
    missing.append('netid')
if class_code.strip() == '':
    missing.append('class_code')
if send_name == '':
    missing.append('nickname')
if len(decision['reason'].strip()) < 30:
    missing.append("decision['reason'] (one or two full sentences)")
if new_mix is None:
    missing.append("a decision that solves (see the cell above)")

sectors = {}
for t in my_stocks:
    sectors[t] = my_stocks[t]
send_limit = None
if new_limit is not None:
    send_limit = float(new_limit)
entry = {
    'netid': netid.strip().lower(),
    'nickname': send_name,
    'theme': send_theme,
    'reason': ("Studio 6 (%s): %s" % (decision['choice'], decision['reason'].strip()))[:200],
    'class_code': class_code.strip(),
    'portfolios': {'no_rules': new_twin, 'with_rules': new_mix},
    'risk_limit': send_limit,
    'rules': rules,
    'extra_rules': extra_rules,
    'stock_theme': my_theme,
    'sectors': sectors,
    'data_window': [str(returns.index[0]), str(returns.index[-1])],
    'decision': decision['choice'],
    'notebook': 'studio6',
}

if len(missing) > 0:
    print("Not sent yet. Fill in:", ", ".join(missing), "then run this cell again.")
elif 'google.colab' not in sys.modules:
    print("Not sent: run this notebook in Google Colab to sign in and submit.")
else:
    from google.colab import auth
    import google.auth
    import google.auth.transport.requests
    auth.authenticate_user()
    credentials, _ = google.auth.default()
    credentials.refresh(google.auth.transport.requests.Request())
    answer = requests.post(SUBMIT_URL, data=json.dumps(entry, default=str), timeout=60,
                           headers={'Authorization': 'Bearer ' + credentials.token, 'Content-Type': 'application/json'})
    receipt = answer.json()
    if receipt.get('ok'):
        received = pd.Timestamp(receipt['received_utc']).tz_convert('America/New_York')
        print("Saved. This is what the leaderboard has:")
        print("  fund:         ", receipt['nickname'])
        print("  NetID:        ", receipt['netid'], " (signed in as", receipt['google_email'] + ")")
        print("  reason:       ", receipt.get('reason', ''))
        print("  received:     ", received.strftime('%a %b %d, %I:%M %p'), "Eastern")
        print("  stocks held:  ", receipt['holdings']['with_rules'])
        print("  it counts from the next market close (4 PM Eastern on a weekday)")
        if receipt.get('record'):
            print("  your private record:", receipt['record'])
            if not has_fund:
                print("     -> paste this as record_link at the top of this notebook, and keep it to yourself")
        print("  the leaderboard: ", BOARD_URL)
    else:
        print("Not accepted:", receipt.get('problem'))

### ✍️ Before you leave tonight (Studio 6 credit)

Double-click and answer. Two or three sentences each, with numbers.

**1. What did the model say today, and how different is it from what I hold?**

*Your answer:*

**2. What did I decide (hold, refresh or change), and what convinced me?**

*Your answer:*

Then save this notebook (File, Download, Download .ipynb) and hand it in on HuskyCT.

# After tonight: you are still the manager

From now until the market closes on **Thursday, Dec 3**, the fund is yours to run.

* **Come back whenever you like**: open this notebook, Runtime, Run all, and read Parts 2 to 4 with the newest prices.
* **Change your fund as often as you like** (or never): set your decision in Part 5 and send it. Each send counts
  from the next close and is saved with its date and your reason.
* **Every change needs a reason you can defend.** Your final report walks through them one by one.

A manager who trades every day is not better than one who trades once. Trading on noise usually loses to holding, and
the next cell will show you which one you turned out to be.

# The final project (individual, due Sunday, Dec 6, 11:59 PM)

The final project is this notebook, run one last time, and a 2-page report.

1. **After the Dec 3 close** (Thursday, 4 PM), open this notebook with your record link at the top and **Run all**.
   The race is over at that close, so the numbers will not change while you write.
2. **Your report numbers.** The next cell prints every number your report needs.
3. **Your fact sheet.** The cell after it makes `my_fund_fact_sheet.png`: page 3 of your report.
4. **The 2-page report** (a PDF on HuskyCT), in five parts:

| Part | Length | What goes in it |
|---|---|---|
| 1 · My fund and my rules | ¼ page | your stocks, your rules, and why you chose them in Studio 5 |
| 2 · What happened before Studio 6 | ½ page | the race up to Nov 18: the stocks that helped and hurt, what your rules did, how you compared with the crowd and the S&P 500 |
| 3 · My decision on Nov 18 | ½ page | what the model said, the size of the trades, what you decided and why (the data, another fund, the news) |
| 4 · What happened after | ½ page | Nov 19 to Dec 3: every later change and its reason, and **your decisions vs never touching the fund** |
| 5 · What I learned | ¼ page | what you would do differently, and what you would hold on Jan 1, 2027 |

Your grade does **not** depend on making money. It depends on how clearly you explain what happened and how well you
defend your decisions.

In [ ]:
def since(line, day):
    # the value of a line from one day to the latest close: (start, end)
    piece = line.loc[day:]
    return float(piece.iloc[0]), float(piece.iloc[-1])


print("REPORT NUMBERS (through the close of %s)" % last_close.date())
print()
print("Part 1 · My fund and my rules")
print("  fund: %s | stock list: %s (%d stocks to choose from) | holds %d today" % (my_name, my_theme, len(base_stocks), len(my_today)))
print("  rules: %s" % rules_in_words(base_rules, base_extra))
print()
print("Part 2 · What happened (from the %s close)" % first_day.strftime('%b %d'))
race_text = "  my fund $%.0f (%+.2f%%) | no-rules twin $%.0f | S&P 500 $%.0f" % (
    my_worth, 100 * (my_worth / 10000 - 1), float(twin_line.iloc[-1]), float(spy_line.iloc[-1]))
if crowd_line is not None:
    race_text = race_text + " | class crowd fund $%.0f" % float(crowd_line.iloc[-1])
print(race_text)
print("  worst drop: my fund %.1f%%, twin %.1f%%, S&P 500 %.1f%%" % (worst_drop(my_line), worst_drop(twin_line), worst_drop(spy_line)))
print("  helped most: " + in_words(helpers, "%s %+.0f"))
print("  at the bottom: " + in_words(hurters, "%s %+.0f"))
if gap >= 0:
    print("  my rules earned $%.0f compared with the no-rules twin" % gap)
else:
    print("  my rules cost $%.0f compared with the no-rules twin" % (-gap))
print()
print("Parts 3 and 4 · My decisions")
studio6_sends = 0
for k in range(len(history)):
    one = history[k]
    if one['day'] > last_close:
        continue
    start_money, end_money = since(my_line, one['day'])
    spy_start, spy_end = since(spy_line, one['day'])
    print("  version %d, from the %s close: %s" % (k + 1, one['day'].strftime('%b %d'), one['reason']))
    print("      since then: my fund %+.2f%%, S&P 500 %+.2f%%" % (100 * (end_money / start_money - 1), 100 * (spy_end / spy_start - 1)))
    if one['notebook'] == 'studio6':
        studio6_sends += 1
waiting = 0
for one in all_versions:
    if one.get('counts_from') is None:
        waiting += 1
        if one.get('notebook') == 'studio6':
            studio6_sends += 1
        print("  sent, counts from the next close: %s" % one.get('reason', ''))
if len(history) > 1 and not stand_in:
    never = float(untouched_line.iloc[-1])
    print("  my decisions vs never touching the fund: $%.0f vs $%.0f, so my decisions are worth %+.0f dollars" % (
        my_worth, never, my_worth - never))
else:
    print("  (no change has started counting yet, so 'my decisions vs never touching the fund' appears after your first one does)")
print()
print("CHECKLIST")
print("  [%s] my record link is pasted at the top" % ('x' if has_fund else ' '))
print("  [%s] at least one decision sent from this notebook (hold counts)" % ('x' if studio6_sends > 0 else ' '))
print("  [%s] the Dec 3 close is in the numbers" % ('x' if last_close >= pd.Timestamp('2026-12-03') else ' '))

### Your fund fact sheet

One page about your fund, made from your own numbers: the race with each of your decisions marked, the stocks that
drove it, and the results. It is page 3 of your report, and something you can show in an interview. In Colab the file
downloads automatically.

In [ ]:
fact = figure(figsize=(11, 8.5))
fact.suptitle(my_name + ' · fund fact sheet', fontsize=18, fontweight='bold', x=0.05, ha='left')
fact.text(0.05, 0.915, '%s list | %s' % (my_theme, rules_in_words(base_rules, base_extra)[:110]), fontsize=9.5, color='dimgray')

race = fact.add_axes([0.07, 0.50, 0.56, 0.36])
race.plot(spy_line.index, spy_line.values, color='black', linewidth=2, label='S&P 500')
if crowd_line is not None:
    race.plot(crowd_line.index, crowd_line.values, color='tab:purple', linewidth=1.5, linestyle=':', label='class crowd fund')
if len(history) > 1 and not stand_in:
    race.plot(untouched_line.index, untouched_line.values, color='gray', linewidth=1.5, linestyle='--', label='if I had never changed it')
race.plot(my_line.index, my_line.values, color='tab:green', linewidth=3, label='my fund')
for k in range(1, len(history)):
    if history[k]['day'] <= last_close:
        race.axvline(history[k]['day'], color='tab:green', linestyle=':', linewidth=1)
race.axhline(10000, color='lightgray', linewidth=1)
race.set_title('$10,000 since the %s close (dotted lines: my decisions)' % first_day.strftime('%b %d'), fontsize=11)
race.xaxis.set_major_locator(mdates.AutoDateLocator(maxticks=6))
race.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
race.legend(fontsize=8, loc='best')

numbers = fact.add_axes([0.66, 0.50, 0.30, 0.36])
numbers.axis('off')
rows_out = [['my fund', '$%.0f' % my_worth, '%.1f%%' % worst_drop(my_line)],
            ['no-rules twin', '$%.0f' % float(twin_line.iloc[-1]), '%.1f%%' % worst_drop(twin_line)],
            ['S&P 500', '$%.0f' % float(spy_line.iloc[-1]), '%.1f%%' % worst_drop(spy_line)]]
if crowd_line is not None:
    rows_out.append(['class crowd fund', '$%.0f' % float(crowd_line.iloc[-1]), '%.1f%%' % worst_drop(crowd_line)])
board = numbers.table(cellText=rows_out, colLabels=['', '$10,000 became', 'worst drop'], loc='upper center', colWidths=[0.40, 0.34, 0.26])
board.auto_set_font_size(False)
board.set_fontsize(8)
board.scale(1, 2.0)
numbers.text(0, 0.18, 'through the close of %s' % last_close.strftime('%b %d, %Y'), fontsize=9, color='dimgray')

drivers = fact.add_axes([0.07, 0.07, 0.38, 0.34])
top = by_stock['dollars added ($)']
top = top.reindex(top.abs().sort_values(ascending=False).index).head(10).sort_values()
colors = []
for t in top.index:
    if top[t] >= 0:
        colors.append('tab:green')
    else:
        colors.append('black')
drivers.barh(top.index, top.values, color=colors)
drivers.axvline(0, color='gray', linewidth=1)
drivers.set_title('The stocks that drove it (dollars)', fontsize=11)
drivers.tick_params(labelsize=8)

log = fact.add_axes([0.52, 0.07, 0.45, 0.34])
log.axis('off')
log.text(0, 1.0, 'My decisions', fontsize=11, fontweight='bold', va='top')
spot = 0.88
for k in range(len(history)):
    text = '%s: %s' % (history[k]['day'].strftime('%b %d'), history[k]['reason'])
    log.text(0, spot, text[:150], fontsize=8, va='top', wrap=True)
    spot -= 0.17
    if spot < 0.2:
        break
log.text(0, 0.05, 'Built with Python, pandas, Pyomo, Bonmin and ipopt: mean-variance optimization with\n'
         'business rules, run as a live fund. OPIM 5641 · UConn · Fall 2026', fontsize=8, color='dimgray', va='bottom')
fact.savefig('my_fund_fact_sheet.png', dpi=150)
show()

print("A resume line you can adjust:")
print("  Built and managed a live stock portfolio in Python (Pyomo, Bonmin, ipopt): mean-variance optimization with "
      "business-rule constraints, re-solved on new data, with every trading decision logged and reviewed against "
      "the S&P 500 (%+.1f%% vs %+.1f%%)." % (100 * (my_worth / 10000 - 1), 100 * (float(spy_line.iloc[-1]) / 10000 - 1)))
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download('my_fund_fact_sheet.png')

### Prompts to try with Claude

Paste one into Claude (or another AI assistant) **together with your own tables**, then check what it says against
the notebook:

* *"Here is my fund's stock-by-stock table. In three sentences, what explains most of my result, and what would I
  check to tell skill from luck over a few weeks?"*
* *"Here is the trade list from re-solving my model with one more month of data. Why would a mean-variance model
  change its mind this much after one month?"*
* *"Here is my decision and my reason. Argue the other side: what is the strongest case for doing the opposite?"*